In [5]:
from z3 import *

field=[[]]

dim=[7,7]
game="2222a3a1a31a2b1b3a2c21223e3a2b1b31a1a2a"
game=sum([[int(x)] if ord(x)<=ord('9') else [0]*int(ord(x)-ord('a')+1) for x in game],[])
for s in game:
    if len(field[-1])==dim[0]:
        field.append([])
    field[-1].append(s)
field[-1]+=[0]*(dim[0]-len(field[-1]))
for x in field:
    print(x)

s = Solver()

Pair = Datatype('Pair')
Pair.declare('make', ('id', IntSort()))
Pair = Pair.create()


h=[[Const("h_%i_%i" %(i,j),IntSort()) for i in range(dim[1])] for j in range(dim[0]+1)]
v=[[Const("v_%i_%i" %(i,j),IntSort()) for i in range(dim[1]+1)] for j in range(dim[0])]

hs=[[Const("hs_%i_%i" %(i,j),IntSort()) for i in range(dim[1])] for j in range(dim[0]+1)]
vs=[[Const("vs_%i_%i" %(i,j),IntSort()) for i in range(dim[1]+1)] for j in range(dim[0])]



def append(a,m,x,y):
    if 0<=x and x<len(m) and 0<=y and y<len(m[x]):
        a.append(m[x][y])

def have_loop(h,v):
    cond = []
    for x in range(dim[0]):
        for y in range(dim[1]):
            val = field[x][y]
            if val>0:
                cond.append(Sum(h[x][y]!=0,h[x+1][y]!=0,v[x][y]!=0,v[x][y+1]!=0)==val)
    for x in range(len(h)):
        for y in range(len(h[x])):
            next_pieces = []
            append(next_pieces,h,x,y-1)
            append(next_pieces,v,x-1,y)
            append(next_pieces,v,x,y)
            append(next_pieces,h,x,y+1)
            append(next_pieces,v,x,y+1)
            append(next_pieces,v,x-1,y+1)

            cond.append(Implies(h[x][y]!=0,   And( Sum([z==h[x][y] for z in next_pieces])==2, 
                                            Sum([z==0 for z in next_pieces])==(len(next_pieces)-2)  )  )) #means that only 2 neightbour is colored and its color equal to current piece

    for x in range(len(v)):
        for y in range(len(v[x])):
            next_pieces = []
            append(next_pieces,h,x+1,y)
            append(next_pieces,h,x+1,y-1)
            append(next_pieces,v,x+1,y)
            append(next_pieces,h,x,y)
            append(next_pieces,h,x,y-1)
            append(next_pieces,v,x-1,y)

            cond.append(Implies(v[x][y]!=0,   And( Sum([z==v[x][y] for z in next_pieces])==2, 
                                            Sum([z==0 for z in next_pieces])==(len(next_pieces)-2)  )  )) #means that only 2 neightbour is colored and its color equal to current piece
    return And(cond)

def is_same_structure(h,hs,v,vs):
    cond = []
    for x in range(len(h)):
        for y in range(len(h[x])):
            cond.append(If(h[x][y]==0,hs[x][y]==0,hs[x][y]!=0))

    for x in range(len(v)):
        for y in range(len(v[x])):
            cond.append(If(v[x][y]==0,vs[x][y]==0,vs[x][y]!=0))
    return And(cond)

vars_hs_vs=[]
for x in hs+vs:
    for e in x:
        vars_hs_vs.append(e)

#if I will remove this lines then z3 will find infinite solutions which are same and just differ by id of color
for x in h+v:
    for e in x:
        s.add(Implies(e!=0,e==1))

#condition to check that only one loop exist
#TODO find sorter way to formalize that only one loop exist
s.add(And(have_loop(h,v),Not(Exists(vars_hs_vs,And(
                                    have_loop(hs,vs),
                                    is_same_structure(h,hs,v,vs),
                                    Or([ e==1 for e in vars_hs_vs]),
                                    Or([ e==2 for e in vars_hs_vs])
)))))



while True:
    st=s.check()
    if st==sat:
        m = s.model()
        for x in range(len(h)):
            for y in range(len(h[x])):
                if m.eval(h[x][y]).as_long()!=0:
                    print("._",end='')
                else:
                    print(". ",end='')
            print(".",end='')
            print()
            if x < len(v):
                for y in range(len(v[x])):
                    if m.eval(v[x][y]).as_long()!=0:
                        print("| ",end='')
                    else:
                        print("  ",end='')
                print()
        s.add(Not(And([ m.eval(h[x][y])==h[x][y] for x in range(len(h)) for y in range(len(h[x]))   ]+
                      [ m.eval(v[x][y])==v[x][y] for x in range(len(v)) for y in range(len(v[x]))   ])))
    else:
        print(st)
        break


[2, 2, 2, 2, 0, 3, 0]
[1, 0, 3, 1, 0, 2, 0]
[0, 1, 0, 0, 3, 0, 2]
[0, 0, 0, 2, 1, 2, 2]
[3, 0, 0, 0, 0, 0, 3]
[0, 2, 0, 0, 1, 0, 0]
[3, 1, 0, 1, 0, 2, 0]
._._. ._._. ._.
|   | |   | | | 
. . . . . ._. .
|   | |       | 
. . ._. ._._._.
|       |       
._._._. ._._._.
      |       | 
._._. ._. ._. .
|   |   | | | | 
._. . ._. . ._.
  | | |   |     
._. ._. . ._._.
|             | 
._._._._._._._.
unsat
